In [36]:
# Instalar bibliotecas necessárias
%pip install scikit-learn pandas numpy matplotlib joblib

Note: you may need to restart the kernel to use updated packages.


# Pipeline de Machine Learning - Dataset de Carros

Este notebook demonstra um pipeline completo de processamento e modelagem para previsão de preços de carros usando o `dataset_tratado.csv`.

**Objetivo:** Prever o preço de carros com base em suas características técnicas (potência, cilindradas, velocidade, etc.).

In [37]:
# Importar bibliotecas necessárias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
import matplotlib.pyplot as plt

# Carregar o dataset tratado
df = pd.read_csv('dataset_tratado.csv')

# Visualizar primeiras linhas e informações
print("Primeiras linhas do dataset:")
print(df.head())
print("\nInformações do dataset:")
print(df.info())
print("\nEstatísticas descritivas:")
print(df.describe())

Primeiras linhas do dataset:
    company                 model       engine      cc     hp  max_speed  \
0      ford                   KA+  1.2L Petrol  1200.0   85.0        165   
1  mercedes               GT 63 S           V8  3982.0  630.0        250   
2      audi            AUDI R8 Gt          V10  5204.0  602.0        320   
3    others            VANTAGE F1           V8  3982.0  656.0        314   
4    others  Continental GT Azure           V8  3996.0  550.0        318   

   0_100_sec      price fuel_type  seats  torque  
0       10.5    78000.0    Petrol      5   140.0  
1        3.2   837200.0    Petrol      4   900.0  
2        3.6  1317108.0    Petrol      2   560.0  
3        3.6  1005888.0    Petrol      2   685.0  
4        4.0  1617200.0    Petrol      4   900.0  

Informações do dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 954 entries, 0 to 953
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  ----- 

## 1. Divisão Inicial dos Dados (Train/Test Split)

A primeira etapa consiste em separar o conjunto de dados em **dados de treino** e **dados de teste**. 

Para o nosso dataset de carros:
- **Features numéricas:** cc, hp, max_speed, 0_100_sec, seats, torque
- **Features categóricas:** company, fuel_type
- **Target (alvo):** price (preço do carro em reais)

Vamos usar uma divisão 80/20 (treino/teste) com `random_state=42` para garantir reprodutibilidade.

In [38]:
# Separar features (X) e target (y)
alvo = 'price'
X = df.drop(columns=[alvo, 'model', 'engine'])  # Removemos model e engine pois são muito específicos
y = df[alvo]

# Divisão em Treino e Teste (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamanho do conjunto de treino: {len(X_train)} amostras")
print(f"Tamanho do conjunto de teste: {len(X_test)} amostras")
print(f"\nColunas utilizadas como features: {list(X.columns)}")

Tamanho do conjunto de treino: 763 amostras
Tamanho do conjunto de teste: 191 amostras

Colunas utilizadas como features: ['company', 'cc', 'hp', 'max_speed', '0_100_sec', 'fuel_type', 'seats', 'torque']


## 2. Identificação e Pré-processamento das Features

Nosso dataset possui dois tipos de features que precisam de tratamento diferente:

### Features Numéricas:
- `cc` - Cilindradas do motor
- `hp` - Potência (cavalos)
- `max_speed` - Velocidade máxima
- `0_100_sec` - Aceleração 0-100 km/h
- `seats` - Número de assentos
- `torque` - Torque do motor

**Tratamento:** Imputação de valores faltantes (mediana) + Escalonamento (StandardScaler)

### Features Categóricas:
- `company` - Marca do carro (já normalizada e agrupada)
- `fuel_type` - Tipo de combustível (Petrol/Diesel/Others)

**Tratamento:** Imputação (mais frequente) + One-Hot Encoding

In [39]:
# Identificação das colunas por tipo de tratamento

# Features NUMÉRICAS (já tratadas no dataset_tratado.csv)
numeric_features = ['cc', 'hp', 'max_speed', '0_100_sec', 'seats', 'torque']

# Features CATEGÓRICAS
categorical_features = ['company', 'fuel_type']

print("Features numéricas:", numeric_features)
print("Features categóricas:", categorical_features)

# Pipeline para features numéricas
# 1. Imputa a mediana (caso ainda haja algum nulo)
# 2. Aplica o escalonamento padrão (média 0, desvio 1)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para features categóricas
# 1. Imputa o valor mais frequente
# 2. Aplica One-Hot Encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))  # Ignora categorias novas no teste
])

# ColumnTransformer para unir os pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("\nPré-processador criado com sucesso!")

Features numéricas: ['cc', 'hp', 'max_speed', '0_100_sec', 'seats', 'torque']
Features categóricas: ['company', 'fuel_type']

Pré-processador criado com sucesso!


## 3. Pipeline Final com Modelo

Agora vamos criar o pipeline completo que integra:
1. **Pré-processamento** (ColumnTransformer)
2. **Modelo de Machine Learning** (Random Forest Regressor)

O pipeline garante que todo o pré-processamento seja aplicado de forma consistente em treino e teste.

In [40]:
# Criar o pipeline completo com o modelo
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

# Treinar o pipeline
# Ao chamar .fit(), o pipeline executa automaticamente:
# 1. fit_transform() do pré-processador em X_train
# 2. fit() do modelo com os dados transformados
model_pipeline.fit(X_train, y_train)

# Avaliar o modelo
# Ao chamar .score(), o pipeline aplica apenas transform() em X_test
score = model_pipeline.score(X_test, y_test)

print(f"Score R² do modelo no conjunto de teste: {score:.4f}")

# Exemplo de predição
new_data = X_test.head(5)
predictions = model_pipeline.predict(new_data)

print("\nExemplos de predições:")
for i, (pred, real) in enumerate(zip(predictions, y_test.head(5))):
    print(f"Carro {i+1}: Predição = R$ {pred:,.2f} | Valor real = R$ {real:,.2f}")

Score R² do modelo no conjunto de teste: 0.9563

Exemplos de predições:
Carro 1: Predição = R$ 127,174.32 | Valor real = R$ 98,800.00
Carro 2: Predição = R$ 467,639.90 | Valor real = R$ 494,000.00
Carro 3: Predição = R$ 153,237.41 | Valor real = R$ 208,000.00
Carro 4: Predição = R$ 143,770.81 | Valor real = R$ 156,000.00
Carro 5: Predição = R$ 228,667.77 | Valor real = R$ 182,000.00


## 4. Tuning de Hiperparâmetros com GridSearchCV

Vamos ajustar os hiperparâmetros do Random Forest para melhorar o desempenho:

- `n_estimators`: número de árvores
- `max_depth`: profundidade máxima
- `min_samples_split`: amostras mínimas para dividir
- `min_samples_leaf`: amostras mínimas em uma folha

Usaremos **validação cruzada 5-fold** para avaliar cada combinação.

In [41]:
# Recriar o pipeline para o Grid Search
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

# Grade de hiperparâmetros
param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4]
}

# GridSearchCV com validação cruzada
grid_search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    n_jobs=-1,  # Usar todos os processadores
    scoring='r2',
    verbose=1
)

print("Iniciando Grid Search...")
grid_search.fit(X_train, y_train)

print("\n" + "="*60)
print("Melhores hiperparâmetros encontrados:")
print(grid_search.best_params_)
print("="*60)

# Avaliar no conjunto de teste
score = grid_search.score(X_test, y_test)
print(f"\nScore R² no conjunto de teste após tuning: {score:.4f}")

# Salvar o melhor modelo
melhor_modelo = grid_search.best_estimator_

Iniciando Grid Search...
Fitting 5 folds for each of 108 candidates, totalling 540 fits

Melhores hiperparâmetros encontrados:
{'regressor__max_depth': 10, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 300}

Score R² no conjunto de teste após tuning: 0.9578

Melhores hiperparâmetros encontrados:
{'regressor__max_depth': 10, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 300}

Score R² no conjunto de teste após tuning: 0.9578


## 5. Comparação de Múltiplos Modelos

Vamos comparar diferentes modelos de regressão para encontrar o melhor:
- Regressão Linear
- Random Forest
- K-Nearest Neighbors (KNN)

In [42]:
# Dicionário com modelos e seus hiperparâmetros
modelos = {
    "Regressão Linear": {
        "modelo": LinearRegression(),
        "param_grid": {}  # Sem hiperparâmetros para tunar
    },
    "Random Forest": {
        "modelo": RandomForestRegressor(random_state=42),
        "param_grid": {
            "regressor__n_estimators": [100, 200, 300],
            "regressor__max_depth": [None, 10, 20],
            "regressor__min_samples_split": [2, 5]
        }
    },
    "KNN": {
        "modelo": KNeighborsRegressor(),
        "param_grid": {
            "regressor__n_neighbors": [3, 5, 7, 9],
            "regressor__weights": ["uniform", "distance"],
            "regressor__p": [1, 2]  # Manhattan ou Euclidiana
        }
    }
}

resultados = []
melhores_modelos = {}

# Loop para treinar e avaliar cada modelo
for nome, info in modelos.items():
    print(f"\nTreinando e ajustando: {nome}...")
    
    modelo = info["modelo"]
    grid = info["param_grid"]
    
    # Criar pipeline
    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', modelo)
    ])
    
    # Grid search (ou treino direto se não tem parâmetros)
    if grid:
        grid_search = GridSearchCV(
            estimator=model_pipeline,
            param_grid=grid,
            cv=5,
            scoring='r2',
            n_jobs=-1,
            verbose=0
        )
        grid_search.fit(X_train, y_train)
        melhores_modelos[nome] = grid_search.best_estimator_
        score = grid_search.best_score_
    else:
        model_pipeline.fit(X_train, y_train)
        from sklearn.model_selection import cross_val_score
        scores = cross_val_score(model_pipeline, X_train, y_train, cv=5, scoring='r2')
        score = scores.mean()
        melhores_modelos[nome] = model_pipeline
    
    resultados.append({
        "Modelo": nome,
        "R² (CV)": score
    })
    print(f"R² médio (validação cruzada): {score:.4f}")

# Exibir tabela comparativa
df_resultados = pd.DataFrame(resultados).sort_values(by="R² (CV)", ascending=False)
print("\n" + "="*60)
print("COMPARAÇÃO DE MODELOS:")
print("="*60)
print(df_resultados.to_string(index=False))


Treinando e ajustando: Regressão Linear...
R² médio (validação cruzada): 0.8747

Treinando e ajustando: Random Forest...
R² médio (validação cruzada): 0.9160

Treinando e ajustando: KNN...
R² médio (validação cruzada): 0.9160

Treinando e ajustando: KNN...
R² médio (validação cruzada): 0.9460

COMPARAÇÃO DE MODELOS:
          Modelo  R² (CV)
             KNN 0.945972
   Random Forest 0.915965
Regressão Linear 0.874673
R² médio (validação cruzada): 0.9460

COMPARAÇÃO DE MODELOS:
          Modelo  R² (CV)
             KNN 0.945972
   Random Forest 0.915965
Regressão Linear 0.874673


## 6. Avaliação do Melhor Modelo no Conjunto de Teste

Agora vamos avaliar o desempenho do melhor modelo encontrado no conjunto de teste (dados não vistos durante o treinamento).

### Intervalos de Confiança

Para dar mais credibilidade científica aos resultados, vamos calcular os **intervalos de confiança (IC 95%)** das métricas usando **validação cruzada 5-fold**. 

O intervalo de confiança indica que temos 95% de confiança de que o verdadeiro valor da métrica está dentro desse intervalo. Isso é importante porque:

1. **Quantifica a incerteza** nas estimativas
2. **Permite comparações mais robustas** entre modelos
3. **É exigido em publicações científicas**

Usamos a **distribuição t de Student** para calcular os intervalos (apropriada para amostras pequenas, como 5 folds).

In [43]:
# Selecionar o melhor modelo
melhor_nome = df_resultados.iloc[0]["Modelo"]
melhor_modelo = melhores_modelos[melhor_nome]

print(f"Melhor modelo: {melhor_nome}")
print("="*60)

# Avaliar no conjunto de teste
score_teste = melhor_modelo.score(X_test, y_test)
print(f"\nR² no conjunto de TESTE: {score_teste:.4f}")

# Fazer predições de exemplo
predictions = melhor_modelo.predict(X_test.head(10))

print("\nExemplos de predições vs valores reais:")
print("-"*60)
for i, (pred, real) in enumerate(zip(predictions, y_test.head(10)), 1):
    erro_pct = abs((pred - real) / real) * 100
    print(f"Carro {i:2d}: Pred = R$ {pred:>11,.2f} | Real = R$ {real:>11,.2f} | Erro = {erro_pct:5.1f}%")

# Calcular métricas com validação cruzada para obter intervalos de confiança
from sklearn.model_selection import cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Validação cruzada 5-fold para calcular intervalos de confiança
cv_results = cross_validate(
    melhor_modelo, 
    X_train, 
    y_train, 
    cv=5,
    scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'],
    return_train_score=False
)

# Converter scores negativos para positivos
mae_cv = -cv_results['test_neg_mean_absolute_error']
rmse_cv = -cv_results['test_neg_root_mean_squared_error']
r2_cv = cv_results['test_r2']

# Calcular médias e intervalos de confiança (95%)
import scipy.stats as stats

def calcular_ic95(valores):
    """Calcula o intervalo de confiança de 95% usando distribuição t de Student"""
    n = len(valores)
    media = np.mean(valores)
    desvio = np.std(valores, ddof=1)
    erro_padrao = desvio / np.sqrt(n)
    # t-crítico para 95% de confiança e n-1 graus de liberdade
    t_critico = stats.t.ppf(0.975, n-1)
    margem_erro = t_critico * erro_padrao
    return media, margem_erro

r2_media, r2_ic = calcular_ic95(r2_cv)
mae_media, mae_ic = calcular_ic95(mae_cv)
rmse_media, rmse_ic = calcular_ic95(rmse_cv)

# Métricas no conjunto de teste (avaliação final)
y_pred_all = melhor_modelo.predict(X_test)
mae_teste = mean_absolute_error(y_test, y_pred_all)
mse_teste = mean_squared_error(y_test, y_pred_all)
rmse_teste = np.sqrt(mse_teste)
r2_teste = r2_score(y_test, y_pred_all)

print("\n" + "="*60)
print("MÉTRICAS COM VALIDAÇÃO CRUZADA (5-fold) - INTERVALO DE CONFIANÇA 95%:")
print("="*60)
print(f"R²:    {r2_media:.4f} ± {r2_ic:.4f}  (IC 95%: [{r2_media-r2_ic:.4f}, {r2_media+r2_ic:.4f}])")
print(f"MAE:   R$ {mae_media:>10,.2f} ± R$ {mae_ic:>10,.2f}")
print(f"RMSE:  R$ {rmse_media:>10,.2f} ± R$ {rmse_ic:>10,.2f}")

print("\n" + "="*60)
print("MÉTRICAS NO CONJUNTO DE TESTE (avaliação final):")
print("="*60)
print(f"R² (coeficiente de determinação):  {r2_teste:.4f}")
print(f"MAE (erro médio absoluto):          R$ {mae_teste:,.2f}")
print(f"RMSE (raiz do erro quadrático):     R$ {rmse_teste:,.2f}")

Melhor modelo: KNN

R² no conjunto de TESTE: 0.9678

Exemplos de predições vs valores reais:
------------------------------------------------------------
Carro  1: Pred = R$  115,682.60 | Real = R$   98,800.00 | Erro =  17.1%
Carro  2: Pred = R$  457,600.00 | Real = R$  494,000.00 | Erro =   7.4%
Carro  3: Pred = R$  285,347.06 | Real = R$  208,000.00 | Erro =  37.2%
Carro  4: Pred = R$  161,912.38 | Real = R$  156,000.00 | Erro =   3.8%
Carro  5: Pred = R$  207,064.22 | Real = R$  182,000.00 | Erro =  13.8%
Carro  6: Pred = R$  315,441.03 | Real = R$  293,280.00 | Erro =   7.6%
Carro  7: Pred = R$  135,327.69 | Real = R$  130,000.00 | Erro =   4.1%
Carro  8: Pred = R$  421,400.31 | Real = R$  543,894.00 | Erro =  22.5%
Carro  9: Pred = R$   50,492.72 | Real = R$   47,840.00 | Erro =   5.5%
Carro 10: Pred = R$   59,280.00 | Real = R$   54,080.00 | Erro =   9.6%

MÉTRICAS COM VALIDAÇÃO CRUZADA (5-fold) - INTERVALO DE CONFIANÇA 95%:
R²:    0.9460 ± 0.0208  (IC 95%: [0.9252, 0.9667])
MAE:

## 7. Visualização: Importância das Features

Para modelos baseados em árvores (Random Forest), podemos visualizar quais features são mais importantes para a predição.

In [44]:
# Verificar se o modelo tem feature_importances_
if hasattr(melhor_modelo.named_steps['regressor'], 'feature_importances_'):
    # Obter importâncias
    importances = melhor_modelo.named_steps['regressor'].feature_importances_
    
    # Obter nomes das features após o pré-processamento
    # Features numéricas mantêm os nomes originais
    feature_names = numeric_features.copy()
    
    # Features categóricas após One-Hot Encoding
    if hasattr(melhor_modelo.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'], 'get_feature_names_out'):
        cat_features = melhor_modelo.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
        feature_names.extend(cat_features)
    
    # Ordenar por importância
    indices = np.argsort(importances)[::-1]
    
    # Plotar
    plt.figure(figsize=(12, 6))
    plt.title(f"Importância das Features - {melhor_nome}", fontsize=14, fontweight='bold')
    plt.bar(range(len(importances)), importances[indices], align="center")
    plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=90)
    plt.xlabel("Features")
    plt.ylabel("Importância")
    plt.tight_layout()
    plt.show()
    
    # Top 10 features mais importantes
    print("\nTop 10 features mais importantes:")
    print("-"*60)
    for i in range(min(10, len(importances))):
        idx = indices[i]
        print(f"{i+1:2d}. {feature_names[idx]:30s} - {importances[idx]:.4f}")
else:
    print(f"O modelo {melhor_nome} não possui feature_importances_")

O modelo KNN não possui feature_importances_


## 8. Salvar o Modelo Final

Vamos salvar o modelo treinado para uso futuro.

In [ ]:
import joblib

# Salvar o modelo completo (pipeline com pré-processamento + modelo)
joblib.dump(melhor_modelo, "modelo_previsao_carros_final.pkl")

print(f"Modelo salvo com sucesso: modelo_previsao_carros_final.pkl")
print(f"Modelo: {melhor_nome}")
print(f"R² no teste: {score_teste:.4f}")
print("\nPara carregar o modelo no futuro, use:")
print("modelo = joblib.load('modelo_previsao_carros_final.pkl')")

Modelo salvo com sucesso: modelo_previsao_carros_final.pkl
Modelo: KNN
R² no teste: 0.9678

Para carregar o modelo no futuro, use:
modelo = joblib.load('modelo_previsao_carros_final.pkl')


: 